# Swahili GPU runner (free Colab or Kaggle T4)

Runs the two GPU steps for the Swahili track: (A) question-answer generation with Gemma, (B) the PnC baseline with AfroXLMR, then scores it.

**Before you start**
1. Upload `swahili_shared_format_upload.zip` (private; it contains corpus text, do not make it public). Colab: put it in your Google Drive. Kaggle: create a *private* dataset from it.
2. Gemma is gated: accept its licence on huggingface.co/google/gemma-3-4b-it, create a read token, and store it as a secret named `HF_TOKEN` (Colab: key icon in the left bar; Kaggle: Add-ons > Secrets). Never paste the token into a cell.
3. Turn on a GPU: Colab *Runtime > Change runtime type > T4 GPU*; Kaggle *Settings > Accelerator > GPU T4 x2* (and Internet on).
Both runs save progress, so if the session stops, run the cells again and they resume.

In [ ]:
import os, sys, glob, zipfile, subprocess
ON_COLAB = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle')
print('Colab' if ON_COLAB else 'Kaggle' if ON_KAGGLE else 'other')
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch the runtime to a GPU')

In [ ]:
# find and unpack the upload
if ON_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    zips = glob.glob('/content/drive/MyDrive/**/swahili_shared_format_upload.zip', recursive=True)
    OUT = '/content/drive/MyDrive/swahili_outputs'
else:
    zips = glob.glob('/kaggle/input/**/swahili_shared_format_upload.zip', recursive=True)
    OUT = '/kaggle/working/swahili_outputs'
assert zips, 'upload swahili_shared_format_upload.zip first'
os.makedirs(OUT, exist_ok=True); zipfile.ZipFile(zips[0]).extractall('/content/sf' if ON_COLAB else '/kaggle/working/sf')
SF = '/content/sf' if ON_COLAB else '/kaggle/working/sf'; print(os.listdir(SF), '->', OUT)

In [ ]:
!pip -q install transformers accelerate bitsandbytes sentencepiece

In [ ]:
# Hugging Face login from a secret (the token is never shown)
if ON_COLAB:
    from google.colab import userdata; tok = userdata.get('HF_TOKEN')
else:
    from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login; login(token=tok, add_to_git_credential=False); print('logged in')

## A. Question-answer generation
Test on a small set first (`--per-domain 20`), read a few pairs, then raise it to 300.

In [ ]:
!cd {SF} && python DataGeneration/generate_qa_swahili.py --data Data.csv --ids Data_row_ids.csv --per-domain 20 --load-4bit --model google/gemma-3-4b-it --out-dir {OUT}/qa_test

In [ ]:
import json
for l in list(open(f'{OUT}/qa_test/qa_pairs_swahili.jsonl', encoding='utf-8'))[:6]:
    r = json.loads(l); print(r['domain'], '|', r['question'], '->', r['answer'], '
')

In [ ]:
# full run (resumable): 300 passages per domain
!cd {SF} && python DataGeneration/generate_qa_swahili.py --data Data.csv --ids Data_row_ids.csv --per-domain 300 --load-4bit --model google/gemma-3-4b-it --out-dir {OUT}/qa

## B. PnC baseline (AfroXLMR)
Start with `--max-train-words 300000 --epochs 2` (about 15 to 30 minutes on a T4); raise it if time allows.

In [ ]:
!cd {SF} && python Modeling/PnC/train_pnc_afroxlmr.py --data Data.csv --eval Evaluation.csv --epochs 2 --max-train-words 300000 --batch 16

In [ ]:
!cd {SF} && python Evaluation/evaluate_pnc.py Evaluation.csv {OUT}/pnc_results.csv && cp Evaluation.csv {OUT}/Evaluation_with_hypothesis.csv

Download `swahili_outputs` (Drive or the Kaggle output panel) and send the `qa` folder, `pnc_results.csv` and `Evaluation_with_hypothesis.csv` back. QA pairs are unreviewed until a native speaker checks a sample.